# 👗 Shop-the-Look: Offline Catalog Indexing

**Run this notebook ONCE on Google Colab with GPU runtime.**

This notebook:
1. Deduplicates 50,000 catalog entries → 28,094 unique products
2. Downloads all product images in parallel (12 workers)
3. Embeds with CLIP ViT-B/32 (GPU-accelerated)
4. Pre-computes attribute labels for the explainer
5. Builds FAISS IndexFlatIP for cosine similarity search

**Expected time:** ~1 hour on Colab T4 GPU

**Output artifacts:** Download and place in `artifacts/` directory of your local repo.

## Step 1: Install Dependencies

In [ ]:
!pip install torch torchvision transformers faiss-gpu Pillow requests tqdm ftfy -q
!mkdir -p artifacts cache/images data src

## Step 2: Upload Dataset Files

Upload `product_catelog.jsonl` and `validation.jsonl` when prompted.

In [ ]:
from google.colab import files
print("Upload product_catelog.jsonl and validation.jsonl")
uploaded = files.upload()

import shutil
for fname in uploaded:
    shutil.move(fname, f"data/{fname}")
    print(f"Moved {fname} → data/{fname}")

## Step 3: Create Source Modules

Write `src/__init__.py`, `src/fetcher.py`, and `src/embedder.py` as Python files.

In [ ]:
# Create src/__init__.py
with open("src/__init__.py", "w") as f:
    f.write("# src package\n")

print("Created src/__init__.py")

In [ ]:
%%writefile src/fetcher.py
"""
src/fetcher.py — Robust image downloader for Pinterest-hosted fashion images.
"""

import os
import time
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_DIR = "cache/images"
os.makedirs(CACHE_DIR, exist_ok=True)


def convert_to_url(signature: str) -> str:
    prefix = 'http://i.pinimg.com/400x/%s/%s/%s/%s.jpg'
    return prefix % (signature[0:2], signature[2:4], signature[4:6], signature)


def fetch_image(signature: str, retries: int = 3, delay: float = 1.5):
    cache_path = os.path.join(CACHE_DIR, f"{signature}.jpg")

    if os.path.exists(cache_path):
        try:
            return Image.open(cache_path).convert("RGB")
        except Exception:
            os.remove(cache_path)

    url = convert_to_url(signature)
    headers = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=10, headers=headers)
            if resp.status_code == 200:
                img = Image.open(BytesIO(resp.content)).convert("RGB")
                img.save(cache_path, format="JPEG", quality=85)
                return img
            elif resp.status_code == 404:
                return None
            else:
                time.sleep(delay * (attempt + 1))
        except requests.exceptions.Timeout:
            time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay * (attempt + 1))

    return None


def parallel_download(signatures: list, max_workers: int = 12) -> dict:
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_sig = {
            executor.submit(fetch_image, sig): sig for sig in signatures
        }
        for future in as_completed(future_to_sig):
            sig = future_to_sig[future]
            try:
                results[sig] = future.result()
            except Exception:
                results[sig] = None
    return results

In [ ]:
%%writefile src/embedder.py
"""
src/embedder.py — CLIP ViT-B/32 image and text embedder.
"""

import torch
import numpy as np
from transformers import CLIPProcessor, CLIPModel

MODEL_NAME = "openai/clip-vit-base-patch32"
EMBED_DIM = 512

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

_model = CLIPModel.from_pretrained(MODEL_NAME).to(device).eval()
_processor = CLIPProcessor.from_pretrained(MODEL_NAME)


def embed_images_batch(pil_images: list, batch_size: int = 64) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(pil_images), batch_size):
        batch = pil_images[i : i + batch_size]
        inputs = _processor(images=batch, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            features = _model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
        all_embeddings.append(features.cpu().numpy())
    return np.vstack(all_embeddings).astype(np.float32)


def embed_single_image(pil_image) -> np.ndarray:
    return embed_images_batch([pil_image])[0]


def embed_texts(texts: list) -> np.ndarray:
    inputs = _processor(
        text=texts, return_tensors="pt", padding=True, truncation=True
    ).to(device)
    with torch.no_grad():
        features = _model.get_text_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy().astype(np.float32)

## Step 4: Create Offline Indexer

In [ ]:
import json
import os
import numpy as np
import faiss
from tqdm.notebook import tqdm
from src.fetcher import fetch_image, parallel_download
from src.embedder import embed_images_batch, embed_texts

CATALOG_FILE = "data/product_catelog.jsonl"
ARTIFACTS_DIR = "artifacts"
CHECKPOINT_EVERY = 500

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Attribute vocabulary for explainer pre-computation
ATTRIBUTES = {
    "color": ["red", "blue", "black", "white", "green", "yellow",
              "pink", "brown", "gray", "beige", "multicolor", "navy",
              "orange", "purple"],
    "pattern": ["solid", "striped", "floral", "checkered", "polka dot",
               "graphic print", "abstract", "animal print", "plaid", "tie-dye"],
    "category": ["dress", "top", "pants", "skirt", "jacket", "shoes",
                "handbag", "accessory", "coat", "shorts", "sweater", "jumpsuit"],
    "style": ["casual", "formal", "sporty", "bohemian", "minimalist",
             "vintage", "streetwear", "elegant", "preppy", "edgy"],
}

## Step 5: Deduplicate Product IDs

In [ ]:
# Deduplicate 50,000 lines → 28,094 unique product IDs
seen = set()
all_ids = []
with open(CATALOG_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        pid = json.loads(line)["product"]
        if pid not in seen:
            seen.add(pid)
            all_ids.append(pid)

print(f"Total lines read: {len(seen) + (50000 - len(all_ids))}")
print(f"Unique product IDs: {len(all_ids)}")  # Expected: 28,094

## Step 6: Download + Embed (with checkpointing)

In [ ]:
# Load checkpoint if available
emb_file = f"{ARTIFACTS_DIR}/checkpoint_embeddings.npy"
ids_file = f"{ARTIFACTS_DIR}/checkpoint_ids.json"

if os.path.exists(emb_file) and os.path.exists(ids_file):
    embeddings = list(np.load(emb_file))
    valid_ids = json.load(open(ids_file))
    start_idx = len(valid_ids)
    print(f"Resuming from checkpoint: {start_idx} already embedded")
else:
    embeddings = []
    valid_ids = []
    start_idx = 0

failed_ids = []
remaining_ids = all_ids[start_idx:]
print(f"Remaining to process: {len(remaining_ids)}")

In [ ]:
# Process in batches of 500 with checkpointing
total_batches = (len(remaining_ids) + CHECKPOINT_EVERY - 1) // CHECKPOINT_EVERY

for batch_start in tqdm(range(0, len(remaining_ids), CHECKPOINT_EVERY),
                        desc="Batches", total=total_batches):
    batch_ids = remaining_ids[batch_start : batch_start + CHECKPOINT_EVERY]

    # Download in parallel
    images_map = parallel_download(batch_ids, max_workers=12)

    # Separate successes from failures
    batch_images, batch_valid_ids = [], []
    for pid in batch_ids:
        img = images_map.get(pid)
        if img is not None:
            batch_images.append(img)
            batch_valid_ids.append(pid)
        else:
            failed_ids.append(pid)

    # Embed successful downloads
    if batch_images:
        batch_embs = embed_images_batch(batch_images, batch_size=64)
        embeddings.extend(batch_embs)
        valid_ids.extend(batch_valid_ids)

    # Save checkpoint
    np.save(f"{ARTIFACTS_DIR}/checkpoint_embeddings.npy",
            np.array(embeddings, dtype=np.float32))
    json.dump(valid_ids, open(f"{ARTIFACTS_DIR}/checkpoint_ids.json", "w"))

    print(f"  Embedded: {len(valid_ids)} | Failed: {len(failed_ids)} | "
          f"Remaining: {len(all_ids) - len(valid_ids) - len(failed_ids)}")

print(f"\nDone! Embedded: {len(valid_ids)}, Failed: {len(failed_ids)}")

## Step 7: Save Final Artifacts

In [ ]:
# Save final embeddings, IDs, and failed list
embeddings_array = np.array(embeddings, dtype=np.float32)
np.save(f"{ARTIFACTS_DIR}/catalog_embeddings.npy", embeddings_array)
json.dump(valid_ids, open(f"{ARTIFACTS_DIR}/catalog_ids.json", "w"))
with open(f"{ARTIFACTS_DIR}/failed_downloads.txt", "w") as f:
    f.write("\n".join(failed_ids))

print(f"Embeddings shape: {embeddings_array.shape}")
print(f"Successfully embedded: {len(valid_ids)}")
print(f"Failed: {len(failed_ids)}")

## Step 8: Pre-compute Attribute Labels

In [ ]:
# Pre-compute attribute classification for explainer
print("Pre-computing attribute vectors for explainer...")

attr_text_embs = {}
for attr, options in ATTRIBUTES.items():
    prompts = [f"a {opt} fashion item" for opt in options]
    attr_text_embs[attr] = (options, embed_texts(prompts))

catalog_attributes = {}
for pid, emb in tqdm(zip(valid_ids, embeddings_array), total=len(valid_ids),
                     desc="Classifying attributes"):
    attrs = {}
    for attr, (options, text_embs) in attr_text_embs.items():
        scores = emb @ text_embs.T
        attrs[attr] = options[int(np.argmax(scores))]
    catalog_attributes[pid] = attrs

json.dump(catalog_attributes, open(f"{ARTIFACTS_DIR}/catalog_attributes.json", "w"))
print(f"Saved attribute labels for {len(catalog_attributes)} products")

## Step 9: Build FAISS Index

In [ ]:
# Build FAISS IndexFlatIP (cosine similarity on L2-normalized vectors)
dim = embeddings_array.shape[1]  # 512
index = faiss.IndexFlatIP(dim)
index.add(embeddings_array)
faiss.write_index(index, f"{ARTIFACTS_DIR}/catalog.index")

print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

## Step 10: Verify Artifacts

In [ ]:
# Verify all artifacts exist and show sizes
print("\nArtifacts:")
for fname in os.listdir(ARTIFACTS_DIR):
    fpath = os.path.join(ARTIFACTS_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname}: {size_mb:.1f} MB")

# Quick sanity check: search for a known product
test_idx = 0
test_emb = embeddings_array[test_idx:test_idx+1]
scores, indices = index.search(test_emb, 1)
print(f"\nSanity check: product {valid_ids[test_idx][:16]}...")
print(f"  Self-similarity score: {scores[0][0]:.4f} (should be ~1.0)")
print(f"  Matched index: {indices[0][0]} (should be {test_idx})")

## Step 11: Download Artifacts

Download all artifact files and place them in the `artifacts/` directory of your local repo.

In [ ]:
from google.colab import files

artifact_files = [
    "catalog_ids.json",
    "catalog_embeddings.npy",
    "catalog.index",
    "catalog_attributes.json",
    "failed_downloads.txt",
]

for fname in artifact_files:
    fpath = f"{ARTIFACTS_DIR}/{fname}"
    if os.path.exists(fpath):
        print(f"Downloading {fname}...")
        files.download(fpath)
    else:
        print(f"WARNING: {fname} not found!")

print("\nAll artifacts downloaded. Place them in your local artifacts/ directory.")

In [ ]:
# Clean up checkpoint files
for ckpt in ["checkpoint_embeddings.npy", "checkpoint_ids.json"]:
    ckpt_path = os.path.join(ARTIFACTS_DIR, ckpt)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
        print(f"Removed checkpoint: {ckpt}")

print("\n✅ Offline indexing complete! You can now run:")
print("   python tune_threshold.py")
print("   python evaluate.py")
print("   python app.py")